In [141]:
restart = True
epoch_to_pickup = 0

In [142]:
# Import libraries

from tensorflow.keras.layers import StringLookup
import numpy as np
import os
import time
import random
import contextlib
import io
import re
import string
import gc  # Import the garbage collector module

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import TextVectorization

In [143]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [144]:
path = ''

In [145]:
# from google.colab import drive
# drive.mount('/content/drive')
# path = '/content/drive/My Drive/M6_Fall2023e/'

## Functions for downloading text


In [146]:
def preprocess_text(text):
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("’", "'").replace("‘", "'")

    # lowercase everything
    text = text.lower()

    # keep letters/numbers/basic punctuation, remove weird junk
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\'\"\-]", " ", text)

    # put spaces around punctuation so punctuation can still be separate tokens
    text = re.sub(r"([.,!?;:\"])"," \1 ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

def postprocess_text(text):
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r'\s+"', ' "', text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [147]:
def postprocess_text(text):

    # Replace special words with whitespace characters
    text = text.replace("zztabzz", "\t")
    text = text.replace("zznewlinezz", "\n")
    text = text.replace("zzspacezz", " ")

    # Remake capital letters at beginning of words
    text = re.sub(r"\^([a-z])", lambda match: f"{match.group(1).upper()}", text)

    text = text.replace("^", "")

    return text

In [148]:
import os

def getMyText():
    local_path = "cleaned_training_text.txt"

    if not os.path.exists(local_path):
        raise FileNotFoundError(
            f"Could not find {local_path}. Put cleaned_training_text.txt in the same folder as this notebook."
        )

    with open(local_path, "rb") as f:
        text = f.read().decode("utf-8", errors="ignore")

    return preprocess_text(text)



In [149]:
getMyText()

'by the general authorities and their families \x01 in response to king ahab s great wickedness \x01 the lord \x01 through the prophet elijah \x01 sealed the heavens \x01 that neither dew nor rain should fall throughout all the land of israel \x01 the drought that ensued and the famine that followed affected elijah himself as well as untold others in the process \x01 ravens did bring elijah bread and meat to eat \x01 but unless ravens carry more than i think they do \x01 this was not a gourmet meal \x01 and ere long the brook cherith \x01 near which he hid and from which he drank \x01 ran dry \x01 and so it went for three years \x01 as the prophet prepared for a final confrontation with ahab \x01 god commanded elijah to go to the village of zarephath \x01 where \x01 he said \x01 he had commanded a widow woman to sustain him \x01 as he entered the city in his weary condition \x01 he met his benefactress \x01 who was undoubtedly as weak and wasted as he \x01 perhaps almost apologetically

In [150]:
def getRandomText(numbooks = 1, verbose=False):
  download_log = io.StringIO()
  text_random = ''
  for b in range(numbooks):
    foundbook = False
    while(foundbook == False):
      booknum = random.randint(100,60000)
      if verbose:
        print('Trying Book #: ',booknum)
      if random.random() > 0.5:
        url = 'https://www.gutenberg.org/files/' + str(booknum) + '/' + str(booknum) + '-0.txt'
        filename_temp = str(booknum) + '-0.txt'
      else:
        url = 'https://www.gutenberg.org/cache/epub/' + str(booknum) + '/pg' + str(booknum) + '.txt'
        filename_temp = 'pg' + str(booknum) + '.txt'
      if verbose:
        print('Trying: ', url)
      try:
        if verbose:
          path_to_file_temp = tf.keras.utils.get_file(filename_temp, url)
        else:
          with contextlib.redirect_stdout(download_log):
            path_to_file_temp = tf.keras.utils.get_file(filename_temp, url)
        temptext = open(path_to_file_temp, 'rb').read().decode(encoding='utf-8')
        tf.io.gfile.remove(path_to_file_temp)
        if (temptext.find('Language: English') >= 0):
          offset = random.randint(-20,20)
          header = 2000
          total_length = 200000
          chopoffend = 10000
          if len(temptext) > (header+total_length+offset+chopoffend):
            foundbook = True
            text_random += temptext[header+offset:header+total_length+offset]
            #print("Yes: " + str(booknum))
            if verbose:
              print('New size of dataset: ', len(text_random))
          elif len(temptext) > (header+12000):
            foundbook = True
            text_random += temptext[header:-chopoffend]
            #print("Yes (smaller): " + str(booknum))
            if verbose:
              print('New size of dataset: ', len(text_random))
          else:
            if verbose:
              print('Not long enough. Trying again...')
            #print("No: " + str(booknum) + " too short")
        else:
          if verbose:
            print('Not English. Trying again...')
          #print("No: " + str(booknum) + " not English")
        del temptext
      except:
        if verbose:
          print('Not valid file. Trying again...')
        #print("No: " + str(booknum) + " not valid")
        foundbook = False
    if verbose:
      print("Found " + str(b+1) + " books so far...")
  del download_log
  #text_random = "".join(c for c in text_random if c in vocab)
  #all_ids_random = ids_from_chars(tf.strings.unicode_split(text_random, 'UTF-8'))
  #ids_dataset_random = tf.data.Dataset.from_tensor_slices(all_ids_random)
  #sequences_random = ids_dataset_random.batch(seq_length+1, drop_remainder=True)
  #dataset_random = sequences_random.map(split_input_target)
  #dataset_random = (dataset_random.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))
  #return dataset_random
  return preprocess_text(text_random)

In [151]:
if restart:
  vocab_text = getMyText()

Make vocabulary (Adapted from TensorFlow word embedding tutorial)

---



In [152]:
# Vocabulary size and number of words in a sequence.
vocab_size = 4000
sequence_length = 60

In [153]:
if restart:
  # Use the text vectorization layer to normalize, split, and map strings to
  # integers. Note that the layer uses the custom standardization defined above.
  # Set maximum_sequence length as all samples are not of the same length.
  vectorize_layer = TextVectorization(
      standardize='lower',
      split='whitespace',
      max_tokens=vocab_size,
      output_mode='int',
      #output_sequence_length=sequence_length
      )

In [154]:
if restart:
  # Make a text-only dataset (no labels) and call adapt to build the vocabulary.
  vectorize_layer.adapt([vocab_text])

In [155]:
if restart:
  vocabulary = vectorize_layer.get_vocabulary()

Save Vocabulary

In [156]:
if restart:
  with open(path + "vocabulary.txt", "w") as file:
    for word in vocabulary:
        file.write(word + "\n")

Load Saved Vocabulary

In [157]:
if restart == False:
  with open(path + "vocabulary.txt", "r") as file:
      vocabulary = [word.strip() for word in file.readlines()]
      vocabulary = vocabulary

  vectorize_layer = TextVectorization(
      vocabulary=vocabulary,
      standardize='lower',
      split='whitespace',
      max_tokens=vocab_size,
      output_mode='int',
      #output_sequence_length=sequence_length
      )

In [158]:
print(vocabulary[:20])
print(vocabulary[-20:])

['', '[UNK]', np.str_('\x01'), np.str_('the'), np.str_('of'), np.str_('and'), np.str_('to'), np.str_('in'), np.str_('that'), np.str_('a'), np.str_('i'), np.str_('we'), np.str_('is'), np.str_('for'), np.str_('he'), np.str_('you'), np.str_('his'), np.str_('be'), np.str_('it'), np.str_('with')]
[np.str_('persuade'), np.str_('personification'), np.str_('persistence'), np.str_('persevere'), np.str_('persecuted'), np.str_('permissiveness'), np.str_('permanent'), np.str_('perils'), np.str_('perilous'), np.str_('peril'), np.str_('performance'), np.str_('perfecting'), np.str_('per'), np.str_('pep'), np.str_('penetrated'), np.str_('peak'), np.str_('peacemakers'), np.str_('pavement'), np.str_('pause'), np.str_('patricia')]


Turn text into a dataset

In [159]:
# This function will generate our sequence pairs:
def split_input_target(sequence):
    input_ids = sequence[:-1]
    target_ids = sequence[1:]
    return input_ids, target_ids

# This function will create the dataset
def text_to_dataset(text):
  all_ids = vectorize_layer(text)
  ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
  del all_ids
  sequences = ids_dataset.batch(sequence_length+1, drop_remainder=True)
  del ids_dataset

  # Call the function for every sequence in our list to create a new dataset
  # of input->target pairs
  dataset = sequences.map(split_input_target)
  del sequences

  # shuffle


  return dataset

Test on vocab text

In [160]:
if restart:
  vocab_ds = text_to_dataset(vocab_text)

In [161]:
def text_from_ids(ids):
  text = ''.join([vocabulary[index] for index in ids])
  return postprocess_text(text)

vocabulary_adjusted = vocabulary
vocabulary_adjusted[0] = '[UNK]'
vocabulary_adjusted[1] = ''

words_from_ids = tf.keras.layers.StringLookup(vocabulary=vocabulary_adjusted, invert=True)

In [162]:
if restart:
  for input_example, target_example in vocab_ds.take(1):
    print("Input: ")
    print(input_example)
    print(text_from_ids(input_example))
    print(words_from_ids(input_example))
    print("Target: ")
    print(target_example)
    print(text_from_ids(target_example))

Input: 
tf.Tensor(
[  51    3  302 2714    5   56  390    2    7  641    6  512 3457   35
  125 2768    2    3   68    2  113    3  173  843    2 2926    3 1119
    2    8  508    1  234 1799  122  456 2012   26    3  712    4  716
    2    3    1    8    1    5    3 1871    8 1334 2737  843  210   20
  137   20 2799  165], shape=(60,), dtype=int64)
bythegeneralauthoritiesandtheirfamiliesinresponsetokingahabsgreatwickednessthelordthroughtheprophetelijahsealedtheheavensthatneithernorrainshouldfallthroughoutallthelandofisraelthethatandthefaminethatfollowedaffectedelijahhimselfaswellasuntoldothers
tf.Tensor(
[b'by' b'the' b'general' b'authorities' b'and' b'their' b'families'
 b'\x01' b'in' b'response' b'to' b'king' b'ahab' b's' b'great'
 b'wickedness' b'\x01' b'the' b'lord' b'\x01' b'through' b'the' b'prophet'
 b'elijah' b'\x01' b'sealed' b'the' b'heavens' b'\x01' b'that' b'neither'
 b'' b'nor' b'rain' b'should' b'fall' b'throughout' b'all' b'the' b'land'
 b'of' b'israel' b'\x01' b'

In [163]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

def setup_dataset(dataset):
  dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))
  return dataset


In [164]:
if restart:
  vocab_ds = setup_dataset(vocab_ds)

## III. Build the model

Next, we'll build our model. Up until this point, you've been using the Keras symbolic, or imperative API for creating your models. Doing something like:

    model = tf.keras.models.Sequentla()
    model.add(tf.keras.layers.Dense(80, activation='relu))
    etc...

However, tensorflow has another way to build models called the Functional API, which gives us a lot more control over what happens inside the model. You can read more about [the differences and when to use each here](https://blog.tensorflow.org/2019/01/what-are-symbolic-and-imperative-apis.html).

We'll use the functional API for our RNN in this example. This will involve defining our model as a custom subclass of `tf.keras.Model`.

If you're not familiar with classes in python, you might want to review [this quick tutorial](https://www.w3schools.com/python/python_classes.asp), as well as [this one on class inheritance](https://www.w3schools.com/python/python_inheritance.asp).

Using a functional model is important for our situation because we're not just training it to predict a single character for a single sequence, but as we make predictions with it, we need it to remember those predictions as use that memory as it makes new predictions.


In [165]:
class ConferenceTextModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim=256, rnn_units=512, dropout_rate=0.2):
        super().__init__()
        self.embedding = layers.Embedding(vocab_size, embedding_dim)

        self.rnn1 = layers.GRU(
            rnn_units,
            return_sequences=True,
            return_state=True,
            dropout=dropout_rate,
            recurrent_dropout=0.0
        )
        self.rnn2 = layers.GRU(
            rnn_units,
            return_sequences=True,
            return_state=True,
            dropout=dropout_rate,
            recurrent_dropout=0.0
        )

        self.dropout = layers.Dropout(dropout_rate)
        self.dense = layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = self.embedding(inputs, training=training)

        if states is None:
            states1 = None
            states2 = None
        else:
            states1, states2 = states

        x, state1 = self.rnn1(x, initial_state=states1, training=training)
        x, state2 = self.rnn2(x, initial_state=states2, training=training)

        x = self.dropout(x, training=training)
        x = self.dense(x, training=training)

        if return_state:
            return x, [state1, state2]
        return x

In [166]:
if restart:
  dataset = vocab_ds
  del vocab_text
  del vocab_ds
else:
  new_text = getRandomText(numbooks = 10)
  dataset = text_to_dataset(new_text)
  del new_text
  dataset = setup_dataset(dataset)

In [167]:
from tensorflow.keras import layers

# Create an instance of our model
#vocab_size=len(ids_from_chars.get_vocabulary())
embedding_dim = 128
rnn_units = 512

model = ConferenceTextModel(vocab_size, embedding_dim, rnn_units)

In [168]:
# Verify the output of our model is correct by running one sample through
# This will also compile the model for us. This step will take a bit.
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")


(64, 60, 4000) # (batch_size, sequence_length, vocab_size)


In [169]:
# Now let's view the model summary
model.summary()

Model: "conference_text_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (64, 60, 128)          │       512,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_18 (GRU)                    │ ((64, 60, 512), (64,   │       986,112 │
│                                 │ 512))                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_19 (GRU)                    │ ((64, 60, 512), (64,   │     1,575,936 │
│                                 │ 512))                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (64, 60, 4000)         │     2,052,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,126,048 (19.55 MB)

 Trainable params: 5,126,048 (19.55 MB)

 Non-trainable params: 0 (0.00 B)

In [170]:
# Here's the code we'll use to sample for us. It has some extra steps to apply
# the temperature to the distribution, and to make sure we don't get empty
# characters in our text. Most importantly, it will keep track of our model
# state for us.

class OneStep(tf.keras.Model):
  def __init__(self, model, vectorize_layer, vocabulary, temperature=0.8, top_p=0.9):
    super().__init__()
    self.temperature=temperature
    self.model = model
    self.vectorize_layer = vectorize_layer
    self.vocabulary = vocabulary
    #print("initialized")

    # Create a mask to prevent "" or "[UNK]" from being generated.
    skip_ids = StringLookup(vocabulary=list(vocabulary))(['', '[UNK]'])[:, None]
    #print(skip_ids)
    #print("3")
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices = skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(vocabulary)])
    #print("4")
    self.prediction_mask = tf.sparse.to_dense(sparse_mask,validate_indices=False)
    #print("5")

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    #input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.vectorize_layer(inputs)
    #print(input_ids)

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits]
    predicted_logits, states =  self.model(inputs=input_ids, states=states,
                                          return_state=True)
    del input_ids
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature

    # Apply the prediction mask: prevent "" or "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    del predicted_logits
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    #print(predicted_ids[0])

    # Return the characters and model state.
    return words_from_ids(predicted_ids), states


In [171]:
def produce_sample(model, vectorize_layer, vocabulary, temp, epoch, prompt):
  # Create an instance of the character generator
  #print("entered")
  one_step_model = OneStep(model, vectorize_layer, vocabulary, temp)
  #print("rand one step")
  # Now, let's generate a 1000 character chapter by giving our model "Chapter 1"
  # as its starting text
  states = None
  next_char = tf.constant([preprocess_text(prompt)])
  result = [tf.constant([prompt])]

  for n in range(200):
    next_char, states = one_step_model.generate_one_step(next_char, states=states)
    #print(next_char)
    result.append(next_char)
    #print(result)

  result = tf.strings.join(result)
  #print(result)

  # Print the results formatted.
  #print('Temp: ' + str(temp) + '\n')
  print(postprocess_text(result[0].numpy().decode('utf-8')))
  #print('\n\n')
  print('Epoch: ' + str(epoch) + '\n', file=open(path + 'tree.txt', 'a'))
  print('Temp: ' + str(temp) + '\n', file=open(path + 'tree.txt', 'a'))
  print(postprocess_text(result[0].numpy().decode('utf-8')), file=open(path + 'tree.txt', 'a'))
  print('\n\n', file=open(path + 'tree.txt', 'a'))
  del states
  del next_char
  del result

## IV. Train the model

For our purposes, we'll be using [categorical cross entropy](https://machinelearningmastery.com/cross-entropy-for-machine-learning/) as our loss function*. Also, our model will be outputting ["logits" rather than normalized probabilities](https://stackoverflow.com/questions/41455101/what-is-the-meaning-of-the-word-logits-in-tensorflow), because we'll be doing further transformations on the output later.


\* Note that since our model deals with integer encoding rather than one-hot encoding, we'll specifically be using [sparse categorical cross entropy](https://stats.stackexchange.com/questions/326065/cross-entropy-vs-sparse-cross-entropy-when-to-use-one-over-the-other).

In [172]:
# sherlock_text = getMyText()

In [173]:
if restart == False:
  model.load_weights(path + "lstm_gru_SH_modelweights_fall2023-random_urls.h5")

In [174]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model = ConferenceTextModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    dropout_rate=0.2
)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer=optimizer, loss=loss)

num_epochs_total = 30

for e in range(num_epochs_total):
    try:
        print(f"\nEpoch {e+1}/{num_epochs_total}")

        new_text = getMyText()
        dataset = text_to_dataset(new_text)
        dataset = setup_dataset(dataset)

        current_lr = 0.001 * (0.97 ** e)
        model.optimizer.learning_rate.assign(current_lr)
        print("Learning rate:", float(model.optimizer.learning_rate.numpy()))

        model.fit(dataset, epochs=1, verbose=1)

        if (e + 1) % 5 == 0:
            print("\n--- Intermediate Sample ---\n")
            # The generate_story function is now called from cell 2bX5y_0gQSbw for final generation.
            # Intermediate samples during training will use produce_sample.

            #print("saving weights...")
            #model.save_weights(path + "lstm_gru_SH_modelweights_fall2023-random_urls.h5")
            #print("weights saved...")
            for temp in [0.2, 0.3, 0.4, 0.5]:
                produce_sample(model, vectorize_layer, vocabulary, temp, e, 'The world seemed like such a peaceful place until the magic tree was discovered in London.')
            print("samples produced...")
            gc.collect()
            print("garbage collected...")
            print("session cleared (to save memory)...")
            #tf.config.experimental.reset_all()
            success = True
    except Exception as ex:
        print(f"An error occurred during epoch {e+1}: {ex}")
        gc.collect()
        #tf.config.experimental.reset_all()
        try:
            del dataset
        except:
            print("dataset already deleted")
        print("retrying epoch: " , e)


Epoch 1/30
Learning rate: 0.0010000000474974513
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - loss: 6.5070

Epoch 2/30
Learning rate: 0.0009699999936856329
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - loss: 5.9423

Epoch 3/30
Learning rate: 0.0009409000049345195
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - loss: 5.9194

Epoch 4/30
Learning rate: 0.0009126730146817863
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - loss: 5.8892

Epoch 5/30
Learning rate: 0.0008852928294800222
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - loss: 5.8330

--- Intermediate Sample ---

The world seemed like such a peaceful place until the magic tree was discovered in London.thethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethethetheandthethethethethethethethethethethethethethethethethethethethethethethethethethetheandthethethethethethethethethethethethethethethethethethethethethethe

In [175]:
import re
import tensorflow as tf
import numpy as np

def generate_story(
    model,
    vocabulary,
    start_string,
    seq_length=80,
    temperature=0.75,
    min_chars=1200,
    top_k=8,
    repetition_penalty=1.6,
    no_repeat_window=25
):
    ids_from_words = tf.keras.layers.StringLookup(
        vocabulary=vocabulary,
        mask_token=None
    )
    words_from_ids = tf.keras.layers.StringLookup(
        vocabulary=ids_from_words.get_vocabulary(),
        invert=True,
        mask_token=None
    )

    words = preprocess_text(start_string).split()

    banned_words = {"[UNK]", "", "-", "--", ".", ",", "!", "?", ";", ":", '"', "''", "``"}

    while len(postprocess_text(" ".join(words))) < min_chars:
        context_words = words[-seq_length:]
        input_ids = ids_from_words(tf.constant(context_words)).numpy().tolist()

        if len(input_ids) < seq_length:
            input_ids = [0] * (seq_length - len(input_ids)) + input_ids

        input_ids = tf.constant([input_ids], dtype=tf.int64)

        logits = model(input_ids, training=False)[:, -1, :].numpy()[0]
        logits = logits / temperature

        recent_words = words[-no_repeat_window:]
        recent_ids = ids_from_words(tf.constant(recent_words)).numpy()

        for token_id in recent_ids:
            if 0 <= token_id < len(logits):
                logits[token_id] /= repetition_penalty

        top_k_indices = np.argpartition(logits, -top_k)[-top_k:]
        top_k_logits = logits[top_k_indices]
        probs = tf.nn.softmax(top_k_logits).numpy()

        chosen_word = None
        for _ in range(50):
            predicted_id = np.random.choice(top_k_indices, p=probs)
            predicted_word = words_from_ids(tf.constant([predicted_id])).numpy()[0].decode("utf-8")

            if predicted_word in banned_words:
                continue
            if re.fullmatch(r"[^\w]+", predicted_word):  # punctuation-only
                continue
            if len(words) > 0 and predicted_word == words[-1]:
                continue

            chosen_word = predicted_word
            break

        if chosen_word is None:
            continue

        words.append(chosen_word)

    text = " ".join(words)
    text = postprocess_text(text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


required_start = "The world seemed like such a peaceful place until the magic tree was discovered in London."
source_note = "This text is based on words from Elder Jeffrey R. Holland."

generated_text = generate_story(
    model=model,
    vocabulary=vocabulary,
    start_string=required_start,
    seq_length=80,
    temperature=0.75,
    min_chars=1200,
    top_k=8,
    repetition_penalty=1.6,
    no_repeat_window=25
)

if not generated_text.startswith(required_start):
    generated_text = required_start + " " + generated_text

final_output = source_note + "\n\n" + generated_text

with open("generated_magic_tree_story.txt", "w", encoding="utf-8") as f:
    f.write(final_output)

print(final_output[:2500])
print("\nCharacter count of story only:", len(generated_text))

This text is based on words from Elder Jeffrey R. Holland.

The world seemed like such a peaceful place until the magic tree was discovered in London. the world seemed like such a peaceful place until the magic tree was discovered in london  and that i will be been are to all with this have can life of a way for the world of the gospel or god from our lord and his be in we was to be that i is all on a father of the life of the church s world with our children for god and his have will be in he are not that i who was been is this man to the church of a father and you were can us for the lord in he are not do it that i have been was is to all or be a world and this love of you for the lord in our church of us with his own way on it was to all is that a father s have can be not you for the lord and this world of the church in us with i will his life is we said that it who know to not be have had a father and the gospel of this church of us or do you in the world for i said from god to no